# Анализ трафика трамваев

### Сторонние данные:

**Historical Weather API (погода в прошлом)**

https://open-meteo.com/en/docs/historical-weather-api/

**Месячный пассажиропоток по всем видам общественного транспорта в городе Москве**

https://data.mos.ru/opendata/62521/

**Moscow Parking Occupancy (4.6M snapshots) (занятость парковок в мск)**

https://www.kaggle.com/datasets/danilmatrosov/moscow-parking-occupancy/

**Russia Holidays (праздники РФ)**

https://pypi.org/project/holidays/


## Область определения

**Город и вид транспорта:** Москва, трамвайные маршруты, представленные в labels_day_train.csv / labels_day_test.csv. В текущем решении обучается 9 маршрутов. Модель не предназначена для метро, автобусов, электробусов или других городов без переобучения.

**Целевая переменная:** почасовое число валидных входов/посадок (boardings) на маршрут. Разрешение - 1 час. 15-минутные кластеры (train_route_flow_15min_clusters.csv) в финальный прогноз не входят и используются как вспомогательный аналитический слой.

**Горизонт:** прогноз с 2025-11-01 00:00 по 2025-12-31 23:00 - примерно 61 день / 1464 часа. Обучение идёт на всех известных метках строго до origin; origin = max(ds) + 1h. Валидация - рекурсивный holdout на labels_day_test.csv, при этом holdout-метки не попадают в признаки, историю или обучение.

**Диапазон данных:** от 2025 года, часовой пояс Europe/Moscow

## Область адаптации

**Новый маршрут:**
- Нужна история почасовых boardings по этому маршруту, в train_models требуется минимум 50 наблюдаемых строк, иначе модель не обучается

- Маршрут нужно добавить в labels, обновить список маршрутов, сезонный профиль и, при необходимости, правила hard-zero

- Переобучить per-route CatBoost и глобальный LightGBM. Для unseen route категориальный route в LightGBM не даёт гарантированной экстраполяции - нужна история или иерархическая/глобальная модель с признаками похожих маршрутов.

**Новый период / год:**
- Обновить календарь: праздники, рабочие выходные, пред-/постпраздничные дни, школьные каникулы

- Обновить TRAM_TRAFFIC_2025_NORM - месячный пассажиропоток для новых месяцев/года

- Подготовить погоду и occupancy на новый период в Europe/Moscow и на часовой сетке

- Дообучить модели на истории до нового origin; при необходимости пересмотреть START/END, ZERO_HOURS, ZERO_ROUTE_IDS, DAMPING_HORIZON_HOURS.

## Пределы применимости
- Только Москва и трамвай. Другие города и виды транспорта вне области определения.

- Из-за рекурсивного прогноза и damping к сезонному профилю после 14 дней (DAMPING_HORIZON_HOURS = 24 * 14) длинный горизонт определяется в основном сезонностью, краткосрочные всплески, погода и occupancy затухают.

- Часы 2–3 принудительно зануляются (ZERO_HOURS=(2, 3)). Маршрут 5 в основном CatBoost-пайплайне также обрабатывается как hard-zero. Если в реальности там есть трафик, модель систематически занижает.

- Weather replay использует ретроспективный CSV. Без available_at нельзя считать, что признаки были доступны в момент прогноза. Для реального применения нужна апишка и подгружать данные в реальном времени

- Веса ансамбля 0.75 / 0.25 и seed-усреднение фиксированы и не выбирались на скрытом тесте. Перенос на новый домен требует повторной валидации.

## Решение

### Предобработка данных

`msk-data-trend.csv` - информация о том какой был пассажиропоток был на том или ином виде транспорта в этом месяце

`occupancy_2025_popular_only.csv` - скластеризированные данные по парковкам, используются только парковки возле
популярных мест: cтадионов, вокзалов, ВДНХ, Москва-Сити, Парк Горького

`open-meteo-msk.csv` - метеорологические данные по погоде в мск в 2025 году

`train_route_flow_15min_clusters` - кластера по типу оплаты (ediny,troyka,koshelek,bank_cards,lgotnye,vedomstvennye,tat,other) на каждом трамвайном маршруте за 15 минут

`holidays.Russia` - информация о праздниках РФ

#### ФИЧИ

| Группа | Признаки |
|---|---|
| Календарные | `hour`, `dayofweek`, `month`, `day`, `dayofyear`, `is_weekend`, `is_day_off`, `is_working_weekend`, `is_holiday`, `is_pre_holiday`, `is_post_holiday`, `is_school_holiday`, `block_len`, `is_long_block_weekend`, `is_nye`, `is_pre_nye`, `day_factor`, `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `tram_traffic_norm` |
| Лаги | `lag_1`, `lag_2`, `lag_3`, `lag_24`, `lag_48`, `lag_72`, `lag_168` |
| Скользящие статистики | `roll_mean_24`, `roll_mean_168`, `roll_std_24` |
| Сезонный профиль маршрута | `route_seasonal_mean` |
| Погода | `temperature_2m`, `rain`, `snowfall`, `snow_depth`, `wind_speed_10m`, `wind_speed_100m` |
| Погодные флаги | `is_rain`, `is_snow`, `temp_below_zero`, `weather_is_real`, `weather_is_partial`, `weather_is_climatology` |
| Occupancy / занятость | `occupancy_rate`, `occupancy_lag_1`, `occupancy_lag_24`, `occupancy_lag_168`, `occupancy_roll_24`, `occupancy_roll_168` |



### Модель

Наше итоговое решение представляет ансамбль. 

Всего в нашем решении обучаются 30 моделей: 27 Catboostов, 3 LightGBM.

Каждый путь (route) мы предсказываем отдельно, используя три модели, которые только отличаются seedом.

У нас SEED = {42, 43, 239}. Потом мы усредняем значения нагруженности полученные тремя этими моделями. Что позволяет получить чуть более качественный результат.

По итогу у нас всего 9 путей => 9*3 = 27 моделей. В нашем коде можно регулировать сиды, в том числе и поставить один, что будет полезно при обучении, когда будет больше 9 путей, так как скорость обучения будет в три раза больше.

Потом все результаты Catboost по путям объединяются в один файл.
Дальше для улучшения метрики я прибег к тому что сделал общую модель для всех путей на LightGBM, тоже с результатами на трех сидах, которые были усреднены. Результат предсказания lightgbm тоже записываются в один файл.

Потом эти результаты объединяются: 

Result = Catboost * 0.75 + LightGBM * 0.25

И это записывается в submit. (5 маршрут всегда нули)


#### Catboost

Для оценки была использована метрика "Quantile:alpha=0.60".

Также были подобраны такие гиперпараметры:
```python
BEST_PARAMS = {
    "iterations": 3000,
    "learning_rate": 0.010,
    "depth": 6,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bagging_temperature": 0.5,
    "border_count": 128,
    "min_data_in_leaf": 20,
    "rsm": 0.7,
    "grow_policy": "SymmetricTree",
    "leaf_estimation_iterations": 3,
    "loss_function": "Quantile:alpha=0.60",
    "random_seed": 42,
    "verbose": False,
    "thread_count": 4,
    "allow_writing_files": False,
}
```

##### Код модели Catboost

In [ ]:
from __future__ import annotations

from collections import Counter
import csv
from datetime import date, timedelta
import hashlib
import json
from pathlib import Path
import re

import holidays
import numpy as np
import pandas as pd

BEST_PARAMS = {
    "iterations": 3000,
    "learning_rate": 0.010,
    "depth": 6,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bagging_temperature": 0.5,
    "border_count": 128,
    "min_data_in_leaf": 20,
    "rsm": 0.7,
    "grow_policy": "SymmetricTree",
    "leaf_estimation_iterations": 3,
    "loss_function": "Quantile:alpha=0.60",
    "random_seed": 42,
    "verbose": False,
    "thread_count": 4,
    "allow_writing_files": False,
}
SEEDS = (42,43,239)
MOSCOW = "Europe/Moscow"
LAGS = (1, 2, 3, 24, 48, 72, 168)
DAMPING_HORIZON_HOURS = 24 * 14
WEATHER_COLUMNS = (
    "temperature_2m", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m",
)
SCHOOL_HOLIDAYS = (
    ("2024-12-30", "2025-01-12"),
    ("2025-03-22", "2025-03-30"),
    ("2025-05-26", "2025-08-31"),
    ("2025-10-25", "2025-11-04"),
)

HOLIDAY_SHORT_FACTOR = 0.48
HOLIDAY_LONG_FACTOR = 0.45
WEEKEND_IN_BLOCK_FACTOR = 0.91
NYE_FACTOR = 0.60
PRE_NYE_FACTOR = 0.93
ZERO_HOURS = (2, 3)
ZERO_ROUTE_IDS = (5,)

CALENDAR_COLUMNS = (
    "hour", "dayofweek", "month", "day", "dayofyear", "is_weekend",
    "is_day_off", "is_working_weekend", "is_holiday", "is_pre_holiday",
    "is_post_holiday", "is_school_holiday",
    "block_len", "is_long_block_weekend", "is_nye", "is_pre_nye",
    "day_factor",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "tram_traffic_norm",
)
OCCUPANCY_PATH = Path("occupancy_2025_popular_only.csv")
OCCUPANCY_MODE = "replay"
OCCUPANCY_LAGS = (1, 24, 168)
OCCUPANCY_FEATURES = (
    "occupancy_rate",
    *(f"occupancy_lag_{lag}" for lag in OCCUPANCY_LAGS),
    "occupancy_roll_24",
    "occupancy_roll_168",
)
FEATURES = (
    *CALENDAR_COLUMNS,
    *(f"lag_{lag}" for lag in LAGS),
    "roll_mean_24", "roll_mean_168", "roll_std_24",
    "route_seasonal_mean",
    *WEATHER_COLUMNS,
    *OCCUPANCY_FEATURES,
    "is_rain", "is_snow", "temp_below_zero",
    "weather_is_real", "weather_is_partial", "weather_is_climatology"
)
TRAM_TRAFFIC_2025_NORM = {
    1: 0.1762,
    2: 0.3781,
    3: 0.7492,
    4: 0.6406,
    5: 0.3391,
    6: 0.0759,
    7: 0.0228,
    8: 0.0000,
    9: 0.5699,
    10: 0.8570,
    11: 0.6845,
    12: 1.0000
}

TRAIN_PATH            = Path("dataset/labels/labels_day_train.csv")
TEST_PATH             = Path("dataset/labels/labels_day_test.csv")
WEATHER_PATH          = Path("open-meteo-msk.csv")
WEATHER_TIMEZONE      = "Europe/Moscow"
WEATHER_MODE          = "replay"
SCHOOL_CALENDAR_PATH  = None 
START                 = "2025-11-01"
END                   = "2025-12-31 23:00:00"
ITERATIONS            = BEST_PARAMS["iterations"]
THREADS               = 4
VALIDATE              = True
VALIDATION_HOURS      = None 
ZERO_ROUTES           = "5"
OUTPUT_PATH           = Path("submission_day3.csv")
REPORT_PATH           = Path("ml_run_report.json")
MODELS_DIR            = Path("model_weights/")


def local_hour(value) -> pd.Timestamp:
    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert(MOSCOW).tz_localize(None)
    if timestamp != timestamp.floor("h"):
        raise ValueError(f"Expected a whole hour: {value}")
    return timestamp


def load_labels(path: Path) -> pd.DataFrame:
    labels = pd.read_csv(path, sep=";")
    required = {"route", "date", "hour", "boardings"}
    if not required.issubset(labels.columns) or labels.empty:
        raise ValueError(f"Invalid labels file: {path}")
    for column in ("route", "hour", "boardings"):
        labels[column] = pd.to_numeric(labels[column], errors="raise")
    if labels[list(required)].isna().any().any():
        raise ValueError("Labels contain missing values")
    if not np.isfinite(labels[["route", "hour", "boardings"]]).all().all():
        raise ValueError("Labels contain nonfinite values")
    if (labels["boardings"] < 0).any():
        raise ValueError("Negative boardings")
    if not labels["hour"].between(0, 23).all():
        raise ValueError("Hour must be in 0..23")
    for column in ("route", "hour"):
        if (labels[column] % 1 != 0).any():
            raise ValueError(f"{column} must be an integer")
        labels[column] = labels[column].astype(int)
    dates = pd.to_datetime(labels["date"], format="%Y-%m-%d", errors="raise")
    labels["ds"] = dates + pd.to_timedelta(labels["hour"], unit="h")
    if labels.duplicated(["route", "ds"]).any():
        raise ValueError("Duplicate route/date/hour labels")
    return labels[["route", "ds", "boardings"]].rename(columns={"boardings": "y"})


def combine_labels(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([train, test], ignore_index=True)
    if combined.duplicated(["route", "ds"]).any():
        raise ValueError("Train/test overlap on route/date/hour")
    return combined.sort_values(["route", "ds"]).reset_index(drop=True)


def load_weather(path: Path, timezone: str | None = None) -> tuple[pd.DataFrame, dict]:
    with path.open(encoding="utf-8-sig", newline="") as source:
        rows = list(csv.reader(source))
    metadata = {}
    if rows and "timezone" in rows[0]:
        metadata = dict(zip(rows[0], rows[1]))
    declared_timezone = metadata.get("timezone")
    if declared_timezone and timezone and timezone != declared_timezone:
        raise ValueError(
            f"Weather timezone {timezone} conflicts with CSV metadata {declared_timezone}"
        )
    timezone = declared_timezone or timezone
    if not timezone:
        raise ValueError(
            "CSV has no timezone metadata; set WEATHER_TIMEZONE to UTC or Europe/Moscow"
        )
    header_position = None
    header = None
    for position, row in enumerate(rows):
        normalized = [name.strip().split(" (", 1)[0] for name in row]
        if {"time", *WEATHER_COLUMNS}.issubset(normalized):
            header_position, header = position, normalized
            break
    if header_position is None:
        raise ValueError("Missing hourly weather table/columns")
    hourly_rows = []
    discarded = 0
    for row in rows[header_position + 1:]:
        if not row:
            continue
        if not re.match(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}", row[0]):
            discarded += 1
            continue
        if len(row) != len(header):
            raise ValueError(f"Malformed hourly weather row: {row[0]}")
        hourly_rows.append(dict(zip(header, row)))
    values = pd.DataFrame(hourly_rows)
    if values.empty:
        raise ValueError("No hourly weather rows")
    timestamps = pd.to_datetime(values["time"], format="ISO8601", errors="raise")
    if timestamps.dt.tz is None:
        timestamps = timestamps.dt.tz_localize(timezone)
    timestamps = timestamps.dt.tz_convert(MOSCOW).dt.tz_localize(None)
    if (timestamps != timestamps.dt.floor("h")).any():
        raise ValueError("Weather timestamps must be whole hours")
    values["ds"] = timestamps
    if values["ds"].duplicated().any():
        raise ValueError("Duplicate hourly weather timestamps")
    for column in WEATHER_COLUMNS:
        values[column] = pd.to_numeric(values[column], errors="coerce")
        values.loc[~np.isfinite(values[column]), column] = np.nan
    if "available_at" in values:
        availability = pd.to_datetime(values["available_at"], format="ISO8601", errors="raise")
        if availability.dt.tz is None:
            availability = availability.dt.tz_localize(timezone)
        values["available_at"] = availability.dt.tz_convert(MOSCOW).dt.tz_localize(None)
    else:
        values["available_at"] = pd.NaT
    report = {
        "hourly_rows": len(values),
        "discarded_non_hourly_rows": discarded,
        "weather_timezone": timezone,
        "timezone_source": "export_metadata" if declared_timezone else "explicit_parameter",
        "export_metadata": metadata,
        "local_from": str(values["ds"].min()),
        "local_to": str(values["ds"].max()),
        "incomplete_hourly_rows": int(values[list(WEATHER_COLUMNS)].isna().any(axis=1).sum()),
    }
    return values.set_index("ds").sort_index(), report

def load_occupancy(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    if "time" not in df.columns or "avg_occupancy_rate" not in df.columns:
        raise ValueError(f"Occupancy file needs 'time' and 'avg_occupancy_rate': {path}")
    ds = pd.to_datetime(df["time"], errors="raise")
    if ds.dt.tz is not None:
        ds = ds.dt.tz_convert(MOSCOW).dt.tz_localize(None)
    if (ds != ds.dt.floor("h")).any():
        raise ValueError("Occupancy timestamps must be whole hours")
    series = pd.Series(
        df["avg_occupancy_rate"].to_numpy(dtype=float), index=ds, name="occupancy_rate"
    )
    return series.groupby(series.index).mean().sort_index()


class Occupancy:
    def __init__(self, series: pd.Series, origin, mode: str = OCCUPANCY_MODE):
        if mode not in {"replay", "causal"}:
            raise ValueError("Occupancy mode must be replay or causal")
        self.series = series
        self.origin = local_hour(origin)
        self.mode = mode
        self.cache: dict = {}

    def _lookup(self, timestamp) -> float:
        return float(self.series.loc[timestamp]) if timestamp in self.series.index else np.nan

    def features(self, timestamp) -> dict:
        timestamp = local_hour(timestamp)
        if timestamp in self.cache:
            return dict(self.cache[timestamp])

        if timestamp >= self.origin and self.mode != "replay":
            current = np.nan
        else:
            current = self._lookup(timestamp)

        lags = {f"occupancy_lag_{lag}": self._lookup(timestamp - pd.Timedelta(hours=lag))
                for lag in OCCUPANCY_LAGS}

        def roll(hours: int) -> float:
            window = self.series.loc[
                (self.series.index >= timestamp - pd.Timedelta(hours=hours))
                & (self.series.index < timestamp)
            ]
            return float(window.mean()) if len(window) else np.nan

        features = {
            "occupancy_rate": current,
            **lags,
            "occupancy_roll_24": roll(24),
            "occupancy_roll_168": roll(168),
        }
        self.cache[timestamp] = features
        return dict(features)

class Calendar:
    def __init__(self, start, end, school_intervals=SCHOOL_HOLIDAYS,
                 tram_traffic=TRAM_TRAFFIC_2025_NORM):
        first = pd.Timestamp(start).date() - timedelta(days=1)
        last = pd.Timestamp(end).date() + timedelta(days=1)
        self.holidays = holidays.Russia(years=range(first.year, last.year + 1))
        self.holiday_dates = set(self.holidays.keys())          # set[datetime.date]
        self.working_weekends = set(self.holidays.weekend_workdays)
        self.school_intervals = [
            (date.fromisoformat(s), date.fromisoformat(e))
            for s, e in school_intervals
        ]
        self.tram_traffic = tram_traffic
        if any(s > e for s, e in self.school_intervals):
            raise ValueError("Invalid school holiday interval")
        self._block_min = first - timedelta(days=45)
        self._block_max = last + timedelta(days=45)
        self._block_cache: dict[date, int] = {}

    def _is_nonwork(self, day: date) -> bool:
        return day.weekday() >= 5 or day in self.holiday_dates

    def block_len(self, day: date) -> int:
        cached = self._block_cache.get(day)
        if cached is not None:
            return cached
        if not self._is_nonwork(day):
            self._block_cache[day] = 0
            return 0
        n = 1
        u = day - timedelta(days=1)
        while u >= self._block_min and self._is_nonwork(u):
            n += 1
            u -= timedelta(days=1)
        u = day + timedelta(days=1)
        while u <= self._block_max and self._is_nonwork(u):
            n += 1
            u += timedelta(days=1)
        self._block_cache[day] = n
        return n

    def day_factor(self, day: date) -> float:
        f = 1.0
        b = self.block_len(day)
        is_h = day in self.holiday_dates
        if is_h and day.weekday() < 5:
            f *= HOLIDAY_SHORT_FACTOR if b <= 4 else HOLIDAY_LONG_FACTOR
        if (not is_h) and day.weekday() >= 5 and b >= 3:
            f *= WEEKEND_IN_BLOCK_FACTOR
        if (day.month, day.day) == (12, 31):
            f = NYE_FACTOR
        elif (day.month, day.day) in ((12, 29), (12, 30)):
            f *= PRE_NYE_FACTOR
        return f

    def features(self, timestamp) -> dict[str, float]:
        timestamp = local_hour(timestamp)
        day = timestamp.date()
        weekend = day.weekday() >= 5
        b = self.block_len(day)
        tram_val = (
            self.tram_traffic.get(timestamp.month, 0.0)
            if timestamp.year == 2025 else 0.0
        )
        return {
            "hour": timestamp.hour,
            "dayofweek": timestamp.dayofweek,
            "month": timestamp.month,
            "day": timestamp.day,
            "dayofyear": timestamp.dayofyear,
            "is_weekend": int(weekend),
            "is_day_off": int(day in self.holiday_dates
                              or (weekend and day not in self.working_weekends)),
            "is_working_weekend": int(day in self.working_weekends),
            "is_holiday": int(day in self.holiday_dates),
            "is_pre_holiday": int(day + timedelta(days=1) in self.holiday_dates),
            "is_post_holiday": int(day - timedelta(days=1) in self.holiday_dates),
            "is_school_holiday": int(any(s <= day <= e for s, e in self.school_intervals)),
            "block_len": b,
            "is_long_block_weekend": int(weekend and (day not in self.holiday_dates) and b >= 3),
            "is_nye": int((day.month, day.day) == (12, 31)),
            "is_pre_nye": int((day.month, day.day) in ((12, 29), (12, 30))),
            "day_factor": self.day_factor(day),
            "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
            "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
            "dow_sin": np.sin(2 * np.pi * timestamp.dayofweek / 7),
            "dow_cos": np.cos(2 * np.pi * timestamp.dayofweek / 7),
            "tram_traffic_norm": float(tram_val),
        }


class Weather:
    def __init__(self, data: pd.DataFrame, origin, mode="replay"):
        if mode not in {"replay", "causal"}:
            raise ValueError("Weather mode must be replay or causal")
        self.data = data
        self.origin = local_hour(origin)
        self.mode = mode
        self.usage = Counter()
        past = data.loc[data.index < self.origin, list(WEATHER_COLUMNS)].copy()
        past["month"] = past.index.month
        past["hour"] = past.index.hour
        self.month_hour = past.groupby(["month", "hour"])[list(WEATHER_COLUMNS)].mean()
        self.hour = past.groupby("hour")[list(WEATHER_COLUMNS)].mean()
        self.global_mean = past[list(WEATHER_COLUMNS)].mean()
        self.cache = {}

    def features(self, timestamp) -> dict[str, float]:
        timestamp = local_hour(timestamp)
        if timestamp in self.cache:
            return dict(self.cache[timestamp])
        actual = pd.Series(np.nan, index=WEATHER_COLUMNS, dtype=float)
        if timestamp in self.data.index:
            row = self.data.loc[timestamp]
            issued_before_origin = pd.notna(row["available_at"]) and row["available_at"] <= self.origin
            if timestamp < self.origin or self.mode == "replay" or issued_before_origin:
                actual = row[list(WEATHER_COLUMNS)].astype(float)
        complete = bool(actual.notna().all())
        partial = bool(actual.notna().any() and not complete)
        filled = actual.copy()
        if (timestamp.month, timestamp.hour) in self.month_hour.index:
            filled = filled.fillna(self.month_hour.loc[(timestamp.month, timestamp.hour)])
        if timestamp.hour in self.hour.index:
            filled = filled.fillna(self.hour.loc[timestamp.hour])
        filled = filled.fillna(self.global_mean)
        features = filled.to_dict()
        features.update({
            "is_rain": self.flag(features["rain"], lambda value: value > 0),
            "is_snow": self.flag(features["snowfall"], lambda value: value > 0),
            "temp_below_zero": self.flag(features["temperature_2m"], lambda value: value < 0),
            "weather_is_real": int(complete),
            "weather_is_partial": int(partial),
            "weather_is_climatology": int((actual.isna() & filled.notna()).any()),
        })
        self.cache[timestamp] = features
        return dict(features)

    @staticmethod
    def flag(value, condition):
        return int(condition(value)) if np.isfinite(value) else np.nan


class SeasonalProfile:
    def __init__(self, labels: pd.DataFrame, origin):
        self.origin = local_hour(origin)
        history = labels.loc[labels["ds"] < self.origin].copy()
        history["dayofweek"] = history["ds"].dt.dayofweek
        history["hour"] = history["ds"].dt.hour
        self.by_route_dow_hour = history.groupby(["route", "dayofweek", "hour"])["y"].mean()
        self.by_route_hour = history.groupby(["route", "hour"])["y"].mean()
        self.by_route = history.groupby("route")["y"].mean()
        self.global_mean = float(history["y"].mean()) if len(history) else 0.0

    def value(self, route: int, timestamp) -> float:
        timestamp = local_hour(timestamp)
        key = (route, timestamp.dayofweek, timestamp.hour)
        if key in self.by_route_dow_hour.index:
            return float(self.by_route_dow_hour.loc[key])
        key = (route, timestamp.hour)
        if key in self.by_route_hour.index:
            return float(self.by_route_hour.loc[key])
        if route in self.by_route.index:
            return float(self.by_route.loc[route])
        return self.global_mean


def build_training_features(labels: pd.DataFrame, calendar: Calendar, weather: Weather, occupancy,
                             profile: SeasonalProfile) -> pd.DataFrame:
    prepared = []
    for route, group in labels.groupby("route"):
        group = group.sort_values("ds").set_index("ds")
        grid = group.reindex(pd.date_range(group.index.min(), group.index.max(), freq="h"))
        grid.index.name = "ds"
        grid["route"] = route
        for lag in LAGS:
            grid[f"lag_{lag}"] = grid["y"].shift(lag)
        previous = grid["y"].shift(1)
        grid["roll_mean_24"] = previous.rolling(24, min_periods=1).mean()
        grid["roll_mean_168"] = previous.rolling(168, min_periods=1).mean()
        grid["roll_std_24"] = previous.rolling(24, min_periods=2).std().fillna(0)
        known = grid.loc[grid["y"].notna()].reset_index()
        prepared.append(known)
    result = pd.concat(prepared, ignore_index=True)
    timestamps = result["ds"].drop_duplicates()
    calendar_rows = {timestamp: calendar.features(timestamp) for timestamp in timestamps}
    weather_rows = {timestamp: weather.features(timestamp) for timestamp in timestamps}
    occupancy_rows = {ts: occupancy.features(ts) for ts in timestamps}
    for column in OCCUPANCY_FEATURES:
        result[column] = result["ds"].map(lambda ts: occupancy_rows[ts][column])
    for column in CALENDAR_COLUMNS:
        result[column] = result["ds"].map(lambda timestamp: calendar_rows[timestamp][column])
    for column in (*WEATHER_COLUMNS, "is_rain", "is_snow", "temp_below_zero",
                   "weather_is_real", "weather_is_partial", "weather_is_climatology"):
        result[column] = result["ds"].map(lambda timestamp: weather_rows[timestamp][column])
    result["route_seasonal_mean"] = [
        profile.value(route, timestamp)
        for route, timestamp in zip(result["route"], result["ds"])
    ]
    return result


def train_models(data: pd.DataFrame, params: dict, seeds=SEEDS) -> dict:
    from catboost import CatBoostRegressor

    models = {}
    for route, group in data.groupby("route"):
        if len(group) < 50:
            raise ValueError(
                f"Route {route}: only {len(group)} observed labels; "
                "refusing silent zero model"
            )
        route_models = []
        for seed in seeds:
            model_params = {**params, "random_seed": int(seed)}
            model = CatBoostRegressor(**model_params)
            model.fit(group[list(FEATURES)], group["y"])
            route_models.append(model)
        models[int(route)] = route_models
        print(
            f"Route {route}: fitted {len(seeds)} models "
            f"(seeds={list(seeds)}) on {len(group)} observed rows",
            flush=True,
        )
    return models


def history_features(history: dict, timestamp) -> dict:
    values = {f"lag_{lag}": history.get(timestamp - pd.Timedelta(hours=lag), np.nan) for lag in LAGS}
    previous = np.array([history.get(timestamp - pd.Timedelta(hours=offset), np.nan) for offset in range(1, 169)])
    known_24 = previous[:24][np.isfinite(previous[:24])]
    known_168 = previous[np.isfinite(previous)]
    values.update({
        "roll_mean_24": float(known_24.mean()) if len(known_24) else np.nan,
        "roll_mean_168": float(known_168.mean()) if len(known_168) else np.nan,
        "roll_std_24": float(known_24.std(ddof=1)) if len(known_24) > 1 else 0,
    })
    return values


def recursive_forecast(models: dict, labels: pd.DataFrame, start, end, calendar: Calendar,
                       weather: Weather, occupancy, profile: SeasonalProfile,
                       zero_routes=()) -> pd.DataFrame:
    start, end = local_hour(start), local_hour(end)
    if start > end or weather.origin > start:
        raise ValueError("Invalid forecast interval/origin")
    observed = labels.loc[labels["ds"] < weather.origin]
    histories = {
        int(route): dict(zip(group["ds"], group["y"]))
        for route, group in observed.groupby("route")
    }
    if not set(models).issubset(histories):
        raise ValueError("A model has no history before the origin")
    zero_routes = set(zero_routes) - set(models)
    hard_zero_routes = set(ZERO_ROUTE_IDS) | zero_routes

    predictions = []
    for timestamp in pd.date_range(weather.origin, end, freq="h"):
        external = {**calendar.features(timestamp), **weather.features(timestamp)}
        if timestamp >= start:
            source = "hourly_csv" if external["weather_is_real"] else "climatology_or_missing"
            weather.usage[source] += 1
        horizon_hours = (timestamp - weather.origin) / pd.Timedelta(hours=1)
        damping = min(1.0, horizon_hours / DAMPING_HORIZON_HOURS)
        hour_is_zero = timestamp.hour in ZERO_HOURS
        for route, route_models in models.items():
            if route in hard_zero_routes:
                prediction = 0.0
            elif hour_is_zero:
                prediction = 0.0
            else:
                seasonal = profile.value(route, timestamp)
                row = {
                    **external,
                    **history_features(histories[route], timestamp),
                    **occupancy.features(timestamp),
                    "route_seasonal_mean": seasonal,
                }
                matrix = np.array([[row[column] for column in FEATURES]], dtype=float)

                seed_predictions = []
                for model in route_models:
                    value = float(
                        np.asarray(model.predict(matrix, thread_count=1)).reshape(-1)[0]
                    )
                    if np.isfinite(value):
                        seed_predictions.append(value)
                if not seed_predictions:
                    raise ValueError(
                        f"All seeds returned nonfinite prediction for {route} at {timestamp}"
                    )
                prediction = max(0.0, float(np.mean(seed_predictions)))

                histories[route][timestamp] = (1 - damping) * prediction + damping * seasonal
            histories.setdefault(route, {})[timestamp] = prediction
            if timestamp >= start:
                predictions.append(
                    {"route": route, "ds": timestamp, "prediction": prediction}
                )
        if timestamp >= start:
            for route in zero_routes - set(models):
                predictions.append({"route": route, "ds": timestamp, "prediction": 0.0})
    result = pd.DataFrame(predictions).sort_values(["route", "ds"]).reset_index(drop=True)
    if result.duplicated(["route", "ds"]).any():
        raise ValueError("Duplicate submission keys")
    return result


def validation_metrics(labels: pd.DataFrame, predictions: pd.DataFrame) -> dict:
    matched = labels.merge(predictions, on=["route", "ds"], validate="one_to_one")
    if len(matched) != len(labels):
        raise ValueError("Validation predictions do not cover all labeled rows")
    error = matched["prediction"].to_numpy() - matched["y"].to_numpy()
    denominator = float(matched["y"].abs().sum())
    return {
        "rows": len(matched),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "wape": float(np.abs(error).sum() / denominator) if denominator else None,
        "validation": "recursive; holdout labels never enter features/history/model",
    }


def write_submission(predictions: pd.DataFrame, output: Path):
    submission = predictions.copy()
    submission["date"] = submission["ds"].dt.strftime("%Y-%m-%d")
    submission["hour"] = submission["ds"].dt.hour
    submission["prediction"] = np.rint(submission["prediction"]).astype(np.int64)
    output.parent.mkdir(parents=True, exist_ok=True)
    submission[["route", "date", "hour", "prediction"]].to_csv(output, sep=";", index=False)


def main():
    if ITERATIONS < 1 or THREADS < 1 or (VALIDATION_HOURS is not None and VALIDATION_HOURS < 1):
        raise SystemExit("iterations/threads/validation-hours must be positive")

    start, end = local_hour(START), local_hour(END)
    train, test = load_labels(TRAIN_PATH), load_labels(TEST_PATH)
    labels = combine_labels(train, test)
    if start <= labels["ds"].max() or start > end:
        raise ValueError("Forecast start must follow all supplied labels; otherwise enable VALIDATE")

    raw_weather, weather_report = load_weather(WEATHER_PATH, WEATHER_TIMEZONE)
    raw_occupancy = load_occupancy(OCCUPANCY_PATH)

    school_intervals = SCHOOL_HOLIDAYS
    if SCHOOL_CALENDAR_PATH:
        school_intervals = json.loads(SCHOOL_CALENDAR_PATH.read_text(encoding="utf-8"))

    calendar = Calendar(labels["ds"].min(), end, school_intervals)
    params = {**BEST_PARAMS, "iterations": ITERATIONS, "thread_count": THREADS}

    report = {
        "inputs": {
            name: {"file": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
            for name, path in (("train", TRAIN_PATH), ("test", TEST_PATH), ("weather", WEATHER_PATH))
        },
        "weather_input": weather_report,
        "weather_mode": WEATHER_MODE,
        "weather_replay_warning": "Hourly CSV is retrospective; without available_at it does not prove operational forecast availability.",
        "school_calendar": "Supplied intervals; school-specific calendar is not independently verified.",
        "params": params,
        "features": list(FEATURES),
        "hard_zero": {"hours": list(ZERO_HOURS), "routes": list(ZERO_ROUTE_IDS)},
        "holiday_factors": {
            "holiday_short": HOLIDAY_SHORT_FACTOR,
            "holiday_long": HOLIDAY_LONG_FACTOR,
            "weekend_in_block": WEEKEND_IN_BLOCK_FACTOR,
            "nye": NYE_FACTOR,
            "pre_nye": PRE_NYE_FACTOR,
        },
    }

    if VALIDATE:
        validation_origin = test["ds"].min().normalize()
        if train["ds"].max() >= validation_origin:
            raise ValueError("Training labels overlap the validation origin")
        validation_end = test["ds"].max()
        if VALIDATION_HOURS:
            validation_end = min(validation_end, validation_origin + pd.Timedelta(hours=VALIDATION_HOURS - 1))
        validation_weather = Weather(raw_weather, validation_origin, WEATHER_MODE)
        validation_profile = SeasonalProfile(train, validation_origin)
        validation_occupancy = Occupancy(raw_occupancy, validation_origin, OCCUPANCY_MODE)
        validation_data = build_training_features(train, calendar, validation_weather, validation_occupancy, validation_profile)
        validation_models = train_models(validation_data, params, SEEDS)
        predictions = recursive_forecast(
            validation_models, train, validation_origin, validation_end, calendar,
            validation_weather, validation_occupancy, validation_profile,
        )
        held_out = test.loc[test["ds"] <= validation_end]
        report["holdout"] = {
            **validation_metrics(held_out, predictions),
            "from": str(validation_origin), "to_inclusive": str(validation_end),
            "weather_usage_hours": dict(validation_weather.usage),
        }
        print("Recursive holdout:", report["holdout"], flush=True)

    origin = labels["ds"].max() + pd.Timedelta(hours=1)
    weather = Weather(raw_weather, origin, WEATHER_MODE)
    occupancy = Occupancy(raw_occupancy, origin, OCCUPANCY_MODE)
    profile = SeasonalProfile(labels, origin)
    training_data = build_training_features(labels, calendar, weather, occupancy, profile)
    models = train_models(training_data, params, SEEDS)
    zero_routes = [int(route) for route in ZERO_ROUTES.split(",") if route.strip()]
    predictions = recursive_forecast(models, labels, start, end, calendar, weather,
                                     occupancy, profile, zero_routes)
    write_submission(predictions, OUTPUT_PATH)

    if MODELS_DIR:
        MODELS_DIR.mkdir(parents=True, exist_ok=True)
        for route, route_models in models.items():
            for idx, model in enumerate(route_models):
                seed = SEEDS[idx]
                model.save_model(str(MODELS_DIR / f"route_{route}_seed{seed}.cbm"))
                model.save_model(
                    str(MODELS_DIR / f"route_{route}_seed{seed}.onnx"),
                    format="onnx",
                )


    report["forecast"] = {
        "origin": str(origin), "from": str(start), "to_inclusive": str(end),
        "rows": len(predictions), "routes": sorted(predictions["route"].unique().tolist()),
        "weather_usage_hours": dict(weather.usage),
        "refit": "train + test labels, all strictly before origin; not a holdout score",
        "output": str(OUTPUT_PATH),
        "occupancy_input": {
            "file": OCCUPANCY_PATH.name if OCCUPANCY_PATH else None,
            "rows": int(len(raw_occupancy)) if OCCUPANCY_PATH else 0,
            "local_from": str(raw_occupancy.index.min()) if len(raw_occupancy) else None,
            "local_to": str(raw_occupancy.index.max()) if len(raw_occupancy) else None,
            "mode": OCCUPANCY_MODE,
        },
    }
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print(f"Saved {len(predictions)} rows to {OUTPUT_PATH}", flush=True)


if __name__ == "__main__":
    main()

#### LightGBM

Код модели

In [ ]:
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import os
import platform
import time
import zipfile
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

SEEDS = (42, 43, 239)
WEATHER = ["temperature_2m", "rain", "snowfall", "snow_depth", "wind_speed_10m", "wind_speed_100m"]
TRAFFIC = {1: .1762, 2: .3781, 3: .7492, 4: .6406, 5: .3391, 6: .0759,
           7: .0228, 8: 0., 9: .5699, 10: .8570, 11: .6845, 12: 1.}
SCHOOL_INTERVALS = [("2024-12-30", "2025-01-12"), ("2025-03-22", "2025-03-30"),
                    ("2025-05-26", "2025-08-31"), ("2025-10-25", "2025-11-04")]
HOLIDAYS_2025 = [f"2025-01-{d:02d}" for d in range(1, 9)] + [
    "2025-02-23", "2025-03-08", "2025-05-01", "2025-05-02", "2025-05-08",
    "2025-05-09", "2025-06-12", "2025-06-13", "2025-11-03", "2025-11-04", "2025-12-31"]
PARAMS = dict(objective="regression_l1", n_estimators=1000, learning_rate=.04,
              num_leaves=31, max_depth=-1, min_child_samples=80,
              colsample_bytree=.95, subsample=1., subsample_freq=0,
              reg_alpha=.2, reg_lambda=8., max_bin=255,
              deterministic=True, force_col_wise=True, verbosity=-1, n_jobs=2)


SELECTED_FEATURES = [
    "route", "hour", "dayofweek", "is_weekend", "is_day_off", "is_working_weekend",
    "is_holiday", "is_pre_holiday", "is_post_holiday", "is_school_holiday", "block_len",
    "day_factor", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "tram_traffic_norm",
]


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def get_bytes(data: Path, filename: str) -> bytes:
    data = Path(data)
    if data.is_file() and data.suffix.lower() == ".zip":
        with zipfile.ZipFile(data) as z:
            matches = [n for n in z.namelist() if Path(n).name == filename]
            if len(matches) != 1:
                raise ValueError(f"Expected one {filename} in {data}, found {matches}")
            return z.read(matches[0])
    candidates = [p for p in data.rglob(filename) if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"Cannot find {filename} under {data}")
    contents = [p.read_bytes() for p in candidates]
    if any(b != contents[0] for b in contents):
        raise ValueError(f"Ambiguous, different copies of {filename} under {data}")
    return contents[0]


def labels_from_bytes(content: bytes) -> pd.DataFrame:
    x = pd.read_csv(io.BytesIO(content), sep=";")
    required = ["route", "date", "hour", "boardings"]
    if not set(required).issubset(x):
        raise ValueError(f"Missing label columns: {required}")
    for col in ["route", "hour", "boardings"]:
        x[col] = pd.to_numeric(x[col], errors="raise")
        if not np.isfinite(x[col]).all():
            raise ValueError(f"Nonfinite label column: {col}")
    if (x["boardings"] < 0).any() or not x.hour.between(0, 23).all():
        raise ValueError("Invalid boardings / hour")
    if ((x[["route", "hour"]] % 1) != 0).any().any():
        raise ValueError("Route and hour must be integers")
    x["route"] = x.route.astype("int32")
    x["ds"] = pd.to_datetime(x.date, format="%Y-%m-%d") + pd.to_timedelta(x.hour, unit="h")
    if x.duplicated(["route", "ds"]).any():
        raise ValueError("Duplicate labels")
    return x[["route", "ds", "boardings"]].rename(columns={"boardings": "y"})


def load_labels(data: Path):
    a = labels_from_bytes(get_bytes(data, "labels_day_train.csv"))
    b = labels_from_bytes(get_bytes(data, "labels_day_test.csv"))
    if a.ds.max() >= b.ds.min().normalize():
        raise ValueError("Expected train strictly before validation origin")
    combined = pd.concat([a, b], ignore_index=True)
    if combined.duplicated(["route", "ds"]).any():
        raise ValueError("Overlapping labels")
    return a, b


def calendar_features(timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    ts = pd.DatetimeIndex(timestamps)
    if not (ts.year == 2025).all():
        raise ValueError("Calendar snapshot is restricted to 2025")
    dates = ts.normalize()
    holidays = pd.DatetimeIndex(HOLIDAYS_2025)
    extended_holidays = holidays.union(pd.date_range("2026-01-01", "2026-01-08")).union(
        pd.DatetimeIndex(["2024-12-30", "2024-12-31"]))
    daygrid = pd.date_range("2024-12-01", "2026-02-15")
    off = pd.Series((daygrid.dayofweek >= 5) | daygrid.isin(extended_holidays), index=daygrid)
    block = off.groupby(off.ne(off.shift()).cumsum()).transform("size").where(off, 0)
    weekend = ts.dayofweek >= 5
    working_weekend = dates == pd.Timestamp("2025-11-01")
    holiday = dates.isin(holidays)
    block_len = block.reindex(dates).to_numpy()
    factors = np.ones(len(ts), dtype=float)
    weekday_holiday = holiday & ~weekend
    factors[weekday_holiday & (block_len <= 4)] = .48
    factors[weekday_holiday & (block_len > 4)] = .45
    factors[(~holiday) & weekend & (block_len >= 3)] *= .91
    nye = (ts.month == 12) & (ts.day == 31)
    pre_nye = (ts.month == 12) & np.isin(ts.day, [29, 30])
    factors[nye] = .60
    factors[pre_nye] *= .93
    school = np.zeros(len(ts), dtype=bool)
    for start, end in SCHOOL_INTERVALS:
        school |= (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))
    frame = pd.DataFrame({
        "hour": ts.hour, "dayofweek": ts.dayofweek, "month": ts.month,
        "day": ts.day, "dayofyear": ts.dayofyear,
        "is_weekend": weekend.astype(int), "is_day_off": (holiday | (weekend & ~working_weekend)).astype(int),
        "is_working_weekend": working_weekend.astype(int), "is_holiday": holiday.astype(int),
        "is_pre_holiday": (dates + pd.Timedelta(days=1)).isin(extended_holidays).astype(int),
        "is_post_holiday": (dates - pd.Timedelta(days=1)).isin(extended_holidays).astype(int),
        "is_school_holiday": school.astype(int), "block_len": block_len,
        "is_long_block_weekend": (weekend & ~holiday & (block_len >= 3)).astype(int),
        "is_nye": nye.astype(int), "is_pre_nye": pre_nye.astype(int), "day_factor": factors,
        "hour_sin": np.sin(2*np.pi*ts.hour/24), "hour_cos": np.cos(2*np.pi*ts.hour/24),
        "dow_sin": np.sin(2*np.pi*ts.dayofweek/7), "dow_cos": np.cos(2*np.pi*ts.dayofweek/7),
        "month_sin": np.sin(2*np.pi*ts.month/12), "month_cos": np.cos(2*np.pi*ts.month/12),
        "tram_traffic_norm": pd.Series(ts.month).map(TRAFFIC).to_numpy(),
    }, index=ts)
    frame.index.name = "ds"
    return frame


def external_features(data: Path) -> pd.DataFrame:
    grid = pd.date_range("2025-01-01", "2025-12-31 23:00", freq="h")
    out = calendar_features(grid)
    rows = list(csv.reader(io.StringIO(get_bytes(data, "open-meteo-msk.csv").decode("utf-8-sig"))))
    for i, row in enumerate(rows):
        names = [v.strip().split(" (", 1)[0] for v in row]
        if {"time", *WEATHER}.issubset(names):
            break
    else:
        raise ValueError("Hourly weather header not found")
    kept = [r for r in rows[i+1:] if len(r) == len(names) and r and len(r[0]) >= 16 and r[0][10:11] == "T"]
    weather = pd.DataFrame(kept, columns=names)
    weather["ds"] = pd.to_datetime(weather.time, errors="raise")
    if weather.ds.dt.tz is not None:
        weather["ds"] = weather.ds.dt.tz_convert("Europe/Moscow").dt.tz_localize(None)
    if weather.duplicated("ds").any():
        raise ValueError("Duplicate weather hours")
    weather = weather.set_index("ds")[WEATHER].apply(pd.to_numeric, errors="coerce")
    out = out.join(weather)
    out["weather_is_real"] = out[WEATHER].notna().all(axis=1).astype("int8")
    for raw, flag in [("rain", "is_rain"), ("snowfall", "is_snow")]:
        out[flag] = (out[raw] > 0).astype(float).where(out[raw].notna())
    out["temp_below_zero"] = (out.temperature_2m < 0).astype(float).where(out.temperature_2m.notna())
    occ = pd.read_csv(io.BytesIO(get_bytes(data, "occupancy_2025_popular_only.csv")))
    ods = pd.to_datetime(occ.time, errors="raise")
    if ods.dt.tz is not None:
        ods = ods.dt.tz_convert("Europe/Moscow").dt.tz_localize(None)
    series = pd.Series(pd.to_numeric(occ.avg_occupancy_rate, errors="raise").to_numpy(), index=ods)
    series = series.groupby(level=0).mean().sort_index()
    occgrid = pd.date_range(min(grid.min()-pd.Timedelta(days=7), series.index.min()),
                            max(grid.max(), series.index.max()), freq="h")
    series = series.reindex(occgrid)
    out["occupancy_rate"] = series.reindex(out.index)
    for lag in [1, 24, 168]:
        out[f"occupancy_lag_{lag}"] = series.shift(lag).reindex(out.index)
    for window in [24, 168]:
        out[f"occupancy_roll_{window}"] = series.shift(1).rolling(window, min_periods=1).mean().reindex(out.index)
    return out.replace([np.inf, -np.inf], np.nan)


def features(keys: pd.DataFrame, external: pd.DataFrame, selected=True) -> pd.DataFrame:
    if not set(keys.ds).issubset(set(external.index)):
        raise ValueError("Prediction keys outside supported 2025 calendar")
    x = external.reindex(pd.DatetimeIndex(keys.ds)).reset_index(drop=True)
    x.insert(0, "route", keys.route.to_numpy(dtype=np.int32))
    return x[SELECTED_FEATURES] if selected else x


def score(y, prediction) -> dict:
    y, prediction = np.asarray(y, dtype=float), np.asarray(prediction, dtype=float)
    if len(y) != len(prediction) or not np.isfinite(prediction).all() or y.sum() <= 0:
        raise ValueError("Invalid scoring input")
    sae = float(np.abs(y - prediction).sum())
    return {"rows": int(len(y)), "absolute_error_sum": sae, "target_sum": float(y.sum()),
            "wape_score": max(0., 1. - sae / float(y.sum()))}


def apply_hard_zeros(prediction, keys):
    prediction = np.maximum(np.asarray(prediction, dtype=float), 0.)
    # Same explicit zero-hour policy as Main. Do NOT zero all rows where Main==0.
    prediction[keys.ds.dt.hour.isin([2, 3]).to_numpy()] = 0.
    if not np.isfinite(prediction).all():
        raise ValueError("LightGBM returned nonfinite predictions")
    return prediction


def save_predictions(keys: pd.DataFrame, prediction, path: Path):
    out = pd.DataFrame({"route": keys.route.to_numpy(), "date": keys.ds.dt.strftime("%Y-%m-%d").to_numpy(),
                        "hour": keys.ds.dt.hour.to_numpy(), "prediction": np.asarray(prediction)})
    if out.duplicated(["route", "date", "hour"]).any():
        raise ValueError("Duplicate prediction keys")
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, sep=";", index=False, float_format="%.17g")


def fit_branch(labels, external, output: Path | None = None):
    x = features(labels, external)
    models = []
    for seed in SEEDS:
        model = lgb.LGBMRegressor(**PARAMS, random_state=seed)
        model.fit(x, labels.y, categorical_feature=["route"])
        models.append(model.booster_)
        if output:
            output.mkdir(parents=True, exist_ok=True)
            model.booster_.save_model(str(output / f"lgbm_seed_{seed}.txt"))
        print(f"LightGBM: seed={seed}, rows={len(labels)}, trees={model.booster_.num_trees()}", flush=True)
    return models


def predict_branch(models, keys, external):
    x = features(keys, external)
    prediction = np.mean([m.predict(x, num_threads=2) for m in models], axis=0)
    return apply_hard_zeros(prediction, keys)


def train(data: Path, output: Path, start="2025-11-01", end="2025-12-31 23:00:00"):
    began = time.perf_counter()
    output.mkdir(parents=True, exist_ok=True)
    a, b = load_labels(data)
    external = external_features(data)
    validation_models = fit_branch(a, external)
    val_pred = predict_branch(validation_models, b, external)
    save_predictions(b, val_pred, output / "lightgbm_validation.csv")
    val_report = {"training_from": str(a.ds.min()), "training_to": str(a.ds.max()),
                  "validation_from": str(b.ds.min()), "validation_to": str(b.ds.max()),
                  "float_predictions": score(b.y, val_pred), "rounded_predictions": score(b.y, np.rint(val_pred))}
    print("Validation (LightGBM ALONE; not Main or blend):", json.dumps(val_report), flush=True)
    # Parameters / seeds / blend weight are not selected using this holdout.
    labels = pd.concat([a, b], ignore_index=True)
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start <= labels.ds.max() or start > end:
        raise ValueError("Submission dates must follow all known labels")
    routes = sorted(int(r) for r in labels.route.unique())
    models = fit_branch(labels, external, output / "models")
    keys = pd.MultiIndex.from_product([routes, pd.date_range(start, end, freq="h")], names=["route", "ds"]).to_frame(index=False)
    prediction = predict_branch(models, keys, external)
    save_predictions(keys, prediction, output / "lightgbm_predictions.csv")
    metadata = {"format_version": 1, "params": PARAMS, "seeds": list(SEEDS), "trained_routes": routes,
                "feature_names": models[0].feature_name(), "label_cutoff_exclusive": str(start),
                "training_max_ds": str(labels.ds.max()), "zero_hours": [2, 3],
                "forecast_start": str(start), "forecast_end": str(end), "rows": len(keys),
                "external_mode": "calendar_and_main_monthly_index", "calibration": "none", "blend_weight": .1,
                "versions": {"python": platform.python_version(), "lightgbm": lgb.__version__, "numpy": np.__version__, "pandas": pd.__version__},
                "inputs": {name: hashlib.sha256(get_bytes(data, name)).hexdigest() for name in ["labels_day_train.csv", "labels_day_test.csv", "open-meteo-msk.csv", "occupancy_2025_popular_only.csv"]},
                "warning": "Selected LightGBM uses calendar and Main's monthly index only. Weather/occupancy candidates were rejected on earlier folds. No hidden-test or blend score measured.",
                "validation_lightgbm_only": val_report, "elapsed_seconds": round(time.perf_counter()-began, 2)}
    (output / "models" / "metadata.json").write_text(json.dumps(metadata, indent=2)+"\n")
    (output / "report.json").write_text(json.dumps(metadata, indent=2)+"\n")
    pd.DataFrame({"feature": models[0].feature_name(), "gain": np.mean([m.feature_importance(importance_type="gain") for m in models], axis=0)}).sort_values("gain", ascending=False).to_csv(output / "feature_importance.csv", index=False)
    print(f"Saved independent predictions to {output / 'lightgbm_predictions.csv'}", flush=True)
    return metadata


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=Path("1.zip"))
    parser.add_argument("--output", type=Path, default=Path("addon_artifacts"))
    parser.add_argument("--start", default="2025-11-01")
    parser.add_argument("--end", default="2025-12-31 23:00:00")
    args = parser.parse_args()
    train(args.data, args.output, args.start, args.end)


if __name__ == "__main__":
    main()


#### Объединение полученных данных

Надо запустить скрипт
```bash
python blend.py --main ./../submission_day3.csv --output submission_main75_lgbm25.csv --weight 0.25
```

In [ ]:
# blend.py
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

KEYS = ["route", "date", "hour"]


def sha256(path: Path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_predictions(path: Path, column="prediction"):
    if not path.is_file():
        raise FileNotFoundError(f"Prediction CSV not found: {path}")
    sample = path.read_text(encoding="utf-8-sig")[:16384]
    sep = csv.Sniffer().sniff(sample, delimiters=";,\t").delimiter
    raw = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if raw.empty or not {*KEYS, column}.issubset(raw):
        raise ValueError(f"{path}: expected route,date,hour,{column}; got {list(raw)}")
    keys = raw[KEYS].copy()
    for c in ["route", "hour"]:
        numbers = pd.to_numeric(keys[c], errors="raise")
        if not np.isfinite(numbers).all() or (numbers % 1 != 0).any():
            raise ValueError(f"Invalid integer {c} in {path}")
        keys[c] = numbers.astype("int64")
    if not keys.hour.between(0, 23).all():
        raise ValueError("Hour outside 0..23")
    keys["date"] = pd.to_datetime(keys.date, format="%Y-%m-%d", errors="raise").dt.strftime("%Y-%m-%d")
    if keys.duplicated(KEYS).any():
        raise ValueError(f"Duplicate prediction keys in {path}")
    p = pd.to_numeric(raw[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(p).all() or (p < 0).any():
        raise ValueError(f"Nonfinite / negative predictions in {path}")
    keys["value"] = p
    return raw, keys, sep


def blend(main_path: Path, addon_path: Path, output: Path, weight=.1,
          main_column="prediction", addon_column="prediction", fallback_routes=(5,), round_output=True):
    main_path, addon_path, output = Path(main_path), Path(addon_path), Path(output)
    if not 0 <= weight <= 1 or not np.isfinite(weight):
        raise ValueError("LightGBM weight must be in [0,1]")
    if output.resolve() in {main_path.resolve(), addon_path.resolve()}:
        raise ValueError("Refusing to overwrite an input CSV")
    before = sha256(main_path)
    raw, main, sep = read_predictions(main_path, main_column)
    _, addon, _ = read_predictions(addon_path, addon_column)
    aligned = main.rename(columns={"value": "main"}).merge(
        addon.rename(columns={"value": "addon"}), on=KEYS, how="left", sort=False,
        validate="one_to_one", indicator=True)
    if len(aligned) != len(main) or not aligned[KEYS].equals(main[KEYS]):
        raise AssertionError("Merge changed Main row order / keys")
    missing = aligned["_merge"].eq("left_only")
    unexpected = missing & ~aligned.route.isin(fallback_routes)
    if unexpected.any():
        raise ValueError("LightGBM is missing keys for a trained/unsupported route. No silent zero-fill. "
                         + str(aligned.loc[unexpected, KEYS].head(5).to_dict("records")))
    aligned.loc[missing, "addon"] = aligned.loc[missing, "main"]
    result = (1. - weight) * aligned.main.to_numpy() + weight * aligned.addon.to_numpy()
    if not np.isfinite(result).all() or (result < 0).any():
        raise ValueError("Invalid blend")
    if round_output:
        result = np.rint(result).astype("int64")
    out = raw.copy()
    out[main_column] = pd.Series(result, index=out.index).astype(str)
    out.loc[missing.to_numpy(), main_column] = raw.loc[missing.to_numpy(), main_column]
    output.parent.mkdir(parents=True, exist_ok=True)
    if weight == 0:
        shutil.copyfile(main_path, output)
    else:
        temp = output.with_name(output.name + ".tmp")
        out.to_csv(temp, sep=sep, index=False)
        temp.replace(output)
    if sha256(main_path) != before:
        raise AssertionError("Main input changed during blending")
    saved, saved_keys, _ = read_predictions(output, main_column)
    if not saved_keys[KEYS].equals(main[KEYS]) or list(saved) != list(raw):
        raise AssertionError("Output key/column preservation failed")
    delta = saved_keys.value.to_numpy() - main.value.to_numpy()
    report = {
        "formula": f"{1-weight:g} * Main + {weight:g} * LightGBM", "round_after_blend": round_output,
        "main_file": str(main_path), "main_sha256_before": before, "main_sha256_after": sha256(main_path),
        "addon_file": str(addon_path), "addon_sha256": sha256(addon_path),
        "output_file": str(output), "output_sha256": sha256(output),
        "rows": len(out), "changed_rows": int(np.count_nonzero(delta)),
        "key_order_preserved": True, "column_order_preserved": True,
        "unchanged_fallback_rows": int(missing.sum()), "fallback_routes": sorted(aligned.loc[missing, "route"].unique().tolist()),
        "mean_absolute_change": float(np.abs(delta).mean()), "max_absolute_change": float(np.abs(delta).max()),
        "total_main": float(main.value.sum()), "total_blend": float(saved_keys.value.sum()),
        "hidden_test_score": None, "validation_blend_score": None,
    }
    output.with_suffix(".audit.json").write_text(json.dumps(report, indent=2)+"\n")
    print(json.dumps(report, indent=2))
    return report


def evaluate(main_path: Path, addon_path: Path, labels_path: Path, output: Path):
    _, a, _ = read_predictions(main_path)
    _, b, _ = read_predictions(addon_path)
    labels = pd.read_csv(labels_path, sep=";")
    labels["date"] = pd.to_datetime(labels.date).dt.strftime("%Y-%m-%d")
    if labels.duplicated(KEYS).any():
        raise ValueError("Duplicate validation labels")
    m = labels.merge(a.rename(columns={"value": "main"}), on=KEYS, how="left", validate="one_to_one").merge(
        b.rename(columns={"value": "addon"}), on=KEYS, how="left", validate="one_to_one")
    if m[["main", "addon"]].isna().any().any():
        raise ValueError("Validation forecasts must cover EVERY labeled row")
    y = pd.to_numeric(m.boardings).to_numpy()
    if y.sum() <= 0:
        raise ValueError("Validation denominator is zero")
    rows = []
    for w in [0., .05, .1, .15, .2, 1.]:
        p = np.rint((1-w)*m.main.to_numpy() + w*m.addon.to_numpy())
        rows.append({"lgbm_weight": w, "wape_score": max(0., 1.-np.abs(y-p).sum()/y.sum()), "rows": len(y)})
    report = pd.DataFrame(rows)
    output.parent.mkdir(parents=True, exist_ok=True)
    report.to_csv(output, index=False)
    print(report.to_string(index=False))
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--main", type=Path, required=True, help="Your original Main CSV, not the previous day4 CSV")
    parser.add_argument("--lgbm", type=Path, default=Path("addon_artifacts/lightgbm_predictions.csv"))
    parser.add_argument("--output", type=Path, default=Path("submission_main90_lgbm10.csv"))
    parser.add_argument("--weight", type=float, default=.1, help="LightGBM weight; Main weight is 1-weight")
    parser.add_argument("--main-column", default="prediction")
    parser.add_argument("--lgbm-column", default="prediction")
    parser.add_argument("--no-round", action="store_true")
    parser.add_argument("--fallback-routes", default="5", help="Only genuinely untrained routes, comma separated")
    parser.add_argument("--validation-labels", type=Path, help="Optional: evaluate provided HOLDOUT forecasts instead of making submission")
    args = parser.parse_args()
    if args.validation_labels:
        evaluate(args.main, args.lgbm, args.validation_labels, args.output)
    else:
        blend(args.main, args.lgbm, args.output, args.weight, args.main_column, args.lgbm_column,
              tuple(int(r.strip()) for r in args.fallback_routes.split(",") if r.strip()), not args.no_round)


if __name__ == "__main__":
    main()


## Подвал

### Обработка данных транзакций

In [ ]:
import pandas as pd
from collections import defaultdict

SUCCESS_CODES = {1}

CLUSTERS = [
    "ediny",
    "troyka",
    "koshelek",
    "bank_cards",
    "lgotnye",
    "vedomstvennye",
    "tat",
    "other",
]
OTHER_IDX = CLUSTERS.index("other")
N_COLS = len(CLUSTERS)

EDINY = {
    "30 дней", "90 дней", "365 дней",
    "30 дней  Пригород", "365 дней Пригор.",
    "3 суток ЕДИНЫЙ", "1 сутки ЕДИНЫЙ",
    "1 ПОЕЗДКА ЕД.", "2 ПОЕЗД. ЕДИНЫЙ", "1 ПОЕЗД. ЕДИНЫЙ",
    "60 поездок ЕД ТC", "20 поездок ЕД ТC", "40 поездок ЕД ТC",
    "2 поездки ЕД ТК", "1 поездка ЕД ТК",
    "60 поездок ЕД ТП",
    "ВБ 90 Единый Ц", "ВБ 30 Единый Ц",
    "30 Единый Ц К", "90 Единый Ц К", "60 Единый Ц К",
    "ВБ 90 Единый П", "ВБ 30 Единый П",
    "90 Единый П К", "60 Единый П К", "30 Единый П К",
    "Бланк ЕД (агент)", "Бланк ЕД (банк)",
    "Бланк ЕД (экиг)", "Бланк ЕД (платн)",
    "1 сутки Пригород", "3 суток Пригород",
    "90 дней Пригород", "30 дней Пригород",
    "365 дней Пригор.", "365дн. Пригор.ДЗ",
    "30 дн. Пригор.ДЗ", "90 дн. Пригор.ДЗ",
    "1 сутки ТК", "3 суток ТК", "3 суток ТК К", "1 сутки ТК К",
    "60 поездок ТК", "60 п. 90 мин ТК", "20 п. 90 мин ТК",
    "40 п. 90 мин ТК", "11 п. 90 мин ТК", "5 п. 90 мин ТК",
    "1 п. 90 мин ТК", "2 п. 90 мин ТК",
    "ВЕСБ М (7 мин)", "ВЕСБ МО (7 мин)",
    "ВЕСБ МОСКВА", "ВЕСБ МО (тип 4)", "ВЕСБ МО (тип 5)",
    "ВЕСБ МО (тип 6)", "ВЕСБ МО с сопров",
    "ВЛБ МОСКВА", "ВЛБ МО",
    "Перезапись",
    "Билет волонтера", "Волонтер  Победы", "Билет журналиста",
    "РЛ студент", "РЛ Федеральный",
    "365 дн.ЕД.ТК ТЕХ",
    "Бланк ЕТК",
    "ТЕСТ 30д Под.",
    "20  ПОЕЗД. ЕДИНЫЙ", "60  ПОЕЗД. ЕДИНЫЙ",
    "БТК", "БТК МГТ БМ",
}

TROYKA = {
    "Бл.Тройка МГТ БН", "Бл.Тройка МГТ пл", "Бл.Тройка МГТ эк",
    "Бл.Тройка МГТ аг",
}

KOSHELEK = {
    "КОШЕЛЕК",
    "Бланк КОШ (экиг)", "Бланк КОШ(агент)",
    "Бланк КОШ (банк)", "БЛАНК КОШ(платн)",
}

BANK_CARDS = {
    "ББК МГТ",
    "СБП",
    "Бланк МЖД-БСК",
}

LGOTNYE = {
    "СКМ МГТ", "СКМ ММ",
    "СКМ с сопр. МГТ", "СКМ с сопр. ММ",
    "СКМО МГТ", "СКМО с сопр. МГТ",
    "30 дней МГТ СКУ ", "90 дней МГТ СКУ",
    "30 дней СКУ", "90 дней СКУ ", "90 дней СКУ",
    "30 дней СКУ Пр", "90 дней СКУ Пр",
    "30дн ММ+МГТ СКУ", "90дн ММ+МГТ СКУ",
    "30 ММ+МГТ СКУ Пр", "90 ММ+МГТ СКУ Пр",
    "Бланк СКУ", "Бланк СКУ МГТ",
    "СКУ тек. мес.", "СКУ тек.м. МГТ",
    "СКУ 30 дней ", "СКУ 30 дней МГТ",
    "СКУ 90 дней", "СКУ на 3 мес.", "СКУ 3 тек.м. МГТ",
    "30 дней МГТ СКС", "90 дней МГТ СКС",
    "30 дней СКС ", "90 дней СКС ",
    "30 дней СКС Пр", "90 дней СКС Пр",
    "30дн ММ+МГТ СКС ", "90дн ММ+МГТ СКС ",
    "30 ММ+МГТ СКС Пр", "90 ММ+МГТ СКС Пр",
    "Бланк СКС", "Бланк СКС МГТ",
    "СКС тек. мес.", "СКС тек.м. МГТ",
    "СКС 30 дней", "СКС 90 дней", "СКС 30 дней МГТ",
    "30дн ММ+МГТ СКА", "90дн ММ+МГТ СКА ",
    "30 ММ+МГТ СКА Пр", "90 ММ+МГТ СКА Пр",
    "30 дней МГТ СКА ", "90 дней МГТ СКА",
    "30 дней СКА", "90 дней СКА ", "90 дней СКА Пр",
    "30 дней СКА Пр", "Бланк СКА",
    "СКА тек. мес.",
    "30дн ММ+МГТ СКО ", "90дн ММ+МГТ СКО",
    "30 ММ+МГТ СКО Пр", "90 ММ+МГТ СКО Пр",
    "30 дней МГТ СКО", "90 дней МГТ СКО",
    "30 дней СКО ", "90 дней СКО", "90 дней СКО Пр",
    "30 дней СКО Пр", "Бланк СКО",
    "СКО 30 дней МГТ", "СКО тек. м. МГТ",
    "30дн ММ+МГТ СКСТ", "90дн ММ+МГТ СКСТ",
    "30 дн. МГТ СК СТ", "90 дн. МГТ СК СТ",
    "30 дней СК СТ", "90 ММ+МГТ СТ Пр",
}

VEDOMSTVENNYE = {
    "СК ГУВД МГТ", "СК ГУВД ММ",
    "СК ГУ МВД МГТ", "СК ГУ МВД ММ",
    "СК УФНС МГТ",
    "СК УТ МВД МГТ", "СК УТ МВД ММ",
    "СК ГУ ФСИН НГПТ", "СК ГУ ФСИН ММ",
    "Суды г.Москвы МГТ", "Суды г.Москвы ММ",
    "Пропуск метро", "Пропуск руковод.", "Пропуск сотр.УВД",
    "БСК дружин. МГТ", "БСК дружин. ММ",
    "СК МО ММ", "СК МО с сопр. ММ",
}

TAT = {
    "30 дней ТАТ", "90 дней ТАТ",
    "ТАТ на 30 дней", "ТАТ 30 дней", "30 дней ТАТ К",
    "ТАТ на 90 дней", "ТАТ 90 дней", "90 дней ТАТ К",
    "ТАТ 365 дней", "ТАТ на 365 дней", "365 дней ТАТ К",
    "365 дней ТАТ",
    "60 дней ТАТ К", "120 дней ТАТ К", "150 дней К",
    "180 дней ТАТ К", "210 дней ТАТ К", "240 дней ТАТ К",
    "270 дней ТАТ К", "300 дней К", "330 дней ТАТ К",
    "30 дней К", "60 дней К", "90 дней К", "120 дней К",
    "180 дней К", "210 дней К", "240 дней К", "270 дней К",
    "300 дней К", "330 дней К", "365 дней К",
    "30 дней  Пригород",
    "90 дней Приг. К", "60 дней Приг. К", "270 дней Приг. К",
    "300 дней Приг. К", "120 дней Приг. К",
    "365 дней Приг. К", "30 дней Приг. К",
    "30 дней ТАТ К",
    "30 МГТ К", "60 МГТ К", "90 МГТ К",
    "30 дней МГТ СКА ",
    "30 дней Подписки", "30дн.Подписки Пр",
    "30 д Автобус А-Б", "Автобус Б 30 дн.", "Автобус А-Б 30 д",
}

def build_good_type_index():
    idx = {}
    for name, mapping in [
        ("ediny", EDINY),
        ("troyka", TROYKA),
        ("koshelek", KOSHELEK),
        ("bank_cards", BANK_CARDS),
        ("lgotnye", LGOTNYE),
        ("vedomstvennye", VEDOMSTVENNYE),
        ("tat", TAT),
    ]:
        cluster_idx = CLUSTERS.index(name)
        for gt in mapping:
            gt_norm = gt.strip()
            if gt_norm in idx and idx[gt_norm] != cluster_idx:
                print(f"[WARN] Конфликт good_type={gt_norm!r}: "
                      f"{CLUSTERS[idx[gt_norm]]} vs {name}")
                continue
            idx[gt_norm] = cluster_idx
    return idx


GT_INDEX = build_good_type_index()

agg = defaultdict(lambda: [0] * N_COLS)

total_rows = 0
kept_rows = 0
lost_bad_dt = 0
lost_bad_route = 0

for chunk in pd.read_csv("dataset/train.csv", chunksize=100_000, sep=";"):
    total_rows += len(chunk)

    chunk = chunk[chunk["validation_result"].isin(SUCCESS_CODES)]
    if chunk.empty:
        continue

    chunk["good_type"] = chunk["good_type"].astype("string").str.strip()

    dt = pd.to_datetime(chunk["tran_date_time"], errors="coerce")
    chunk = chunk.assign(dt_15=dt.dt.floor("15min"))

    bad_dt = chunk["dt_15"].isna()
    bad_route = chunk["ngpt_route"].isna()
    lost_bad_dt += int(bad_dt.sum())
    lost_bad_route += int(bad_route.sum())

    chunk = chunk[~bad_dt & ~bad_route]
    if chunk.empty:
        continue

    kept_rows += len(chunk)

    grouped = chunk.groupby(
        ["ngpt_route", "dt_15", "good_type"], dropna=False
    ).size()

    for (route, t, gt), cnt in grouped.items():
        if pd.isna(gt):
            idx = OTHER_IDX
        else:
            idx = GT_INDEX.get(str(gt), OTHER_IDX)
        agg[(route, t)][idx] += int(cnt)

print(f"Всего строк:            {total_rows:,}")
print(f"Успешных валидаций:     {kept_rows + lost_bad_dt + lost_bad_route:,}")
print(f"  с битой датой:        {lost_bad_dt:,}")
print(f"  с NaN ngpt_route:     {lost_bad_route:,}")
print(f"  попало в агрегацию:   {kept_rows:,}")

columns = ["route", "dt_15min"] + CLUSTERS
rows = [(r, t, *counts) for (r, t), counts in agg.items()]

result = (
    pd.DataFrame(rows, columns=columns)
      .sort_values(["route", "dt_15min"])
      .reset_index(drop=True)
)
result["total"] = result[CLUSTERS].sum(axis=1)

print("Строк до заливки нулями:", len(result))
print("Сумма total:", result["total"].sum())
assert result["total"].sum() == kept_rows, "Потеряли строки!"

full_range = pd.date_range(
    result["dt_15min"].min(),
    result["dt_15min"].max(),
    freq="15min",
)
routes = result["route"].unique()

print(f"Маршрутов: {len(routes):,}")
print(f"15-мин слотов: {len(full_range):,}")
print(f"Строк после заливки будет: {len(routes) * len(full_range):,}")

full_idx = pd.MultiIndex.from_product(
    [routes, full_range], names=["route", "dt_15min"]
).to_frame(index=False)

result = (
    full_idx
    .merge(result, on=["route", "dt_15min"], how="left")
    .fillna({c: 0 for c in CLUSTERS + ["total"]})
)
result["total"] = result[CLUSTERS].sum(axis=1)

print("Итоговых строк:", len(result))

result.to_csv("train_route_flow_15min_clusters.csv", index=False)
print("Сохранено в route_flow_15min_clusters.csv")
print(result.head())

### Обработка парковок

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('occupancy.parquet')

parking_ids = [
    28, 32, 33, 39, 71, 83, 1905059,
    35, 36, 53, 134,
    94, 59,
    34,
    51,
    89,
    40, 49, 144, 145, 146, 1433906, 1433907,
    72, 164, 17,
    63, 64, 65, 66, 84,
    136, 167,
    126, 127, 411596,
    158
]

PARKING_ID_COL = 'parking_id'

df = df[df[PARKING_ID_COL].isin(parking_ids)].copy()

df['time'] = pd.to_datetime(df['time'], utc=True)

df_grouped = (
    df.groupby('time', as_index=False)['occupancy_rate']
      .mean()
      .rename(columns={'occupancy_rate': 'avg_occupancy_rate'})
)

df_hourly = (
    df_grouped
    .set_index('time')
    .resample('1h')['avg_occupancy_rate']
    .mean()
    .reset_index()
)

df_hourly['time'] = (
    df_hourly['time']
      .dt.tz_convert('Europe/Moscow')
      .dt.tz_localize(None)
)

def seasonal_fill(df, col='avg_occupancy_rate',
                  min_valid=0.0, max_valid=99.0,
                  z_local=4.0, z_seasonal=3.5,
                  window=48, pad=12,
                  noise_frac=0.3, random_state=42):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.set_index('time')

    s = df[col].astype(float)

    s = s.where((s >= min_valid) & (s <= max_valid), np.nan)

    tmp = pd.DataFrame({'y': s})
    tmp['hour'] = tmp.index.hour
    tmp['dow'] = tmp.index.dayofweek

    exp = tmp.groupby(['dow', 'hour'])['y'].transform('median')
    mad_s = (tmp['y'] - exp).abs().groupby([tmp['dow'], tmp['hour']]).transform('median')
    robust_s = 1.4826 * mad_s.replace(0, np.nan)
    z_s = (tmp['y'] - exp).abs() / robust_s
    s = s.where(z_s <= z_seasonal, np.nan)

    med = s.rolling(window, center=True, min_periods=12).median()
    mad = (s - med).abs().rolling(window, center=True, min_periods=12).median()
    robust_std = 1.4826 * mad.replace(0, np.nan)
    z_l = (s - med).abs() / robust_std
    s = s.where(z_l <= z_local, np.nan)

    if pad > 0:
        bad = s.isna()
        bad_padded = (
            bad.rolling(2 * pad + 1, center=True, min_periods=1)
               .max()
               .astype(bool)
        )
        s = s.where(~bad_padded, np.nan)

    tmp = pd.DataFrame({'y': s})
    tmp['hour'] = tmp.index.hour
    tmp['dow'] = tmp.index.dayofweek

    seasonal = tmp.groupby(['dow', 'hour'])['y'].transform('median')
    seasonal_hour = tmp.groupby('hour')['y'].transform('median')
    seasonal = seasonal.fillna(seasonal_hour)

    s_filled = tmp['y'].fillna(seasonal)

    if noise_frac > 0:
        noise_std = (s - seasonal).std()
        if pd.notna(noise_std) and noise_std > 0:
            rng = np.random.default_rng(random_state)
            noise = rng.normal(0, noise_std * noise_frac, size=len(s_filled))
            noise = pd.Series(noise, index=s_filled.index)
            noise = noise.where(tmp['y'].isna(), 0.0)
            s_filled = s_filled + noise

    s_filled = s_filled.clip(min_valid, max_valid)

    df[col] = s_filled
    return df.reset_index()


full_idx = pd.date_range(
    df_hourly['time'].min().floor('h'),
    df_hourly['time'].max().ceil('h'),
    freq='1h',
    name='time'
)

df_hourly = (
    df_hourly
    .set_index('time')
    .reindex(full_idx)
    .rename_axis('time')
)

df_hourly = seasonal_fill(
    df_hourly,
    col='avg_occupancy_rate',
    min_valid=0.0,
    max_valid=99.0,
    z_local=4.0,
    z_seasonal=3.5,
    window=48,
    pad=24,
    noise_frac=0.3,
)

n_missing = df_hourly['avg_occupancy_rate'].isna().sum()
print(f'Пропусков после заполнения: {n_missing}')
print(df_hourly['avg_occupancy_rate'].describe())

mask_2025 = (df_hourly['time'] >= '2025-01-01') & (df_hourly['time'] < '2026-01-01')
df_2025_real = df_hourly.loc[mask_2025].reset_index(drop=True)

print('\ndf_2025_real:', df_2025_real.shape)
print(df_2025_real.head())
print(df_2025_real.tail())

mask_2026 = (df_hourly['time'] >= '2026-01-01') & (df_hourly['time'] <= '2026-03-30 23:00:00')
df_2026 = df_hourly.loc[mask_2026].reset_index(drop=True)

print('\ndf_2026:', df_2026.shape)

df_2025_synth = df_2026.copy()
df_2025_synth['time'] = df_2025_synth['time'].apply(lambda t: t.replace(year=2025))

print('\ndf_2025_synth:', df_2025_synth.shape)

df_2026.to_csv('occupancy_2026.csv', index=False)
df_2026.to_parquet('occupancy_2026.parquet', index=False)

df_2025_synth.to_csv('occupancy_2025_synthetic.csv', index=False)
df_2025_synth.to_parquet('occupancy_2025_synthetic.parquet', index=False)

df_2025_real.to_csv('occupancy_2025_real.csv', index=False)
df_2025_real.to_parquet('occupancy_2025_real.parquet', index=False)

df_all = (
    pd.concat(
        [df_2025_real.assign(source='2025_real'),
         df_2025_synth.assign(source='2025_synthetic')],
        ignore_index=True
    )
    .sort_values('time')
    .reset_index(drop=True)
)
df_all.to_csv('occupancy_real.csv', index=False)

print('\ndf_all:', df_all.shape)
print(df_all['source'].value_counts())
df_2025 = (
    pd.concat(
        [df_2025_real.assign(source='2025_real'),
         df_2025_synth.assign(source='2025_synthetic')],
        ignore_index=True
    )
    .sort_values('time')
    .reset_index(drop=True)
)

df_2025.to_csv('occupancy_2025_popular_only.csv', index=False)
df_2025.to_parquet('occupancy_2025_popular_only.parquet', index=False)

print('df_2025:', df_2025.shape)
print(df_2025['source'].value_counts())
print(df_2025.head())
print(df_2025.tail())